# Exercices XP Gold
Dernière mise à jour : 8 mai 2025

👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment créer des outils interactifs multipages avec Gradio Blocks
Comment gérer l'état de session sur plusieurs composants dans Streamlit
Comment créer une gestion dynamique des entrées/sorties basée sur une logique définie par l'utilisateur
Comment orchestrer la communication frontend et backend à l'aide de FastAPI et Gradio ou Streamlit


🛠️ Ce que vous allez créer
Un mini assistant avec des blocs Gradio et plusieurs flux logiques
Un chatbot Streamlit persistant avec basculement de thème
Une architecture basée sur une API utilisant FastAPI + Gradio
Un formulaire de commentaires avec logique conditionnelle dans Streamlit


# 1. Analyse de l’énoncé
Objectifs globaux

Maîtriser Gradio Blocks pour concevoir des interfaces multipages modulaires et réactives.

Savoir gérer l’état de session et l’interactivité complexe dans Streamlit.

Implémenter une logique métier dynamique pour traiter les entrées/sorties utilisateurs.

Orchestrer la communication frontend/backend avec FastAPI comme serveur d’API et Gradio ou Streamlit en interface.

# 2. Approche détaillée par exercice
Exercice 1 : Gradio Blocks – Assistant multipage avec onglets
Analyse

Construction d’une interface modulaire (trois onglets, trois logiques différentes).

Besoin d’un état persistant partagé (nom utilisateur → gr.State).

Utilisation des composants : gr.Blocks, gr.Tab, gr.Textbox, gr.Dropdown, etc.

Points critiques

Gestion fluide du passage d’un onglet à l’autre, tout en gardant l’état du nom.

Cloisonner les logiques métiers par onglet, mais garder une structure unifiée (fonctionnel, pas de fuite d’état).

Approche de résolution

Initialiser gr.Blocks, créer trois gr.Tab.

Onglet 1 : Textbox pour le nom, bouton de validation, affichage message personnalisé (utilisation de gr.State).

Onglet 2 : Deux champs numériques (a, b), dropdown pour l’opération (+, -, *, /), bouton de calcul, affichage du résultat.

Onglet 3 : Textbox pour le texte, bouton d’analyse, affichage du nombre de mots et de caractères.

Bonus : Stockage du nom dans gr.State, utilisé pour personnaliser chaque onglet si besoin.

In [ ]:
# Cellule 1 : Installer gradio si besoin
# !pip install gradio --upgrade

import gradio as gr

# On utilise gr.State pour partager le nom dans les onglets.
def accueil(nom, state):
    state = nom if nom else state
    if state:
        return f"Bonjour {state} !", state
    else:
        return "Veuillez saisir votre nom.", state

def calculatrice(a, b, operation):
    try:
        a, b = float(a), float(b)
        if operation == "+":
            res = a + b
        elif operation == "-":
            res = a - b
        elif operation == "*":
            res = a * b
        elif operation == "/":
            res = a / b if b != 0 else "Division par zéro !"
        else:
            res = "Opération inconnue."
        return f"Résultat : {res}"
    except Exception as e:
        return f"Erreur : {e}"

def analyseur_texte(texte):
    nb_mots = len(texte.split())
    nb_car = len(texte)
    return f"Nombre de mots : {nb_mots}, caractères : {nb_car}"

with gr.Blocks() as demo:
    state = gr.State("")

    with gr.Tabs():
        # Onglet 1 : Accueil
        with gr.Tab("Accueil"):
            nom_input = gr.Textbox(label="Votre nom")
            bouton_accueil = gr.Button("Valider")
            output_accueil = gr.Textbox(label="Message d'accueil")
            bouton_accueil.click(accueil, [nom_input, state], [output_accueil, state])

        # Onglet 2 : Calculatrice
        with gr.Tab("Calculatrice"):
            a = gr.Textbox(label="A")
            b = gr.Textbox(label="B")
            operation = gr.Dropdown(choices=["+", "-", "*", "/"], value="+", label="Opération")
            bouton_calc = gr.Button("Calculer")
            output_calc = gr.Textbox(label="Résultat")
            bouton_calc.click(calculatrice, [a, b, operation], output_calc)

        # Onglet 3 : Analyseur de texte
        with gr.Tab("Analyseur de texte"):
            texte = gr.Textbox(label="Texte à analyser")
            bouton_analyse = gr.Button("Analyser")
            output_analyse = gr.Textbox(label="Analyse")
            bouton_analyse.click(analyseur_texte, texte, output_analyse)

demo.launch()


# Exercice 2 : Streamlit – Chatbot persistant avec bascule de thème
Analyse

Utilisation de st.session_state pour persistance de l’historique.

Ajout d’une logique de thème (clair/sombre) par bouton radio.

Simulation d’une réponse asynchrone/différée (time.sleep).

Export de l’historique conversationnel (téléchargement .txt).

Points critiques

Correction des problèmes courants d’état de session (initialisation propre).

Gestion de la réactivité du thème (Streamlit n’a pas de mode sombre natif personnalisable, mais peut changer le CSS).

Interface de chat simple mais fonctionnelle.

Approche de résolution

Initialiser l’historique dans st.session_state.

Bascule de thème par radio + mise à jour dynamique (option CSS pour personnalisation avancée si besoin).

Interface chat : champ d’entrée, bouton envoyer, affichage alterné utilisateur/bot.

Réponse différée avec time.sleep avant d’afficher la réponse du bot.

Export : bouton qui télécharge l’historique sous forme de .txt.

In [ ]:
# Cellule 2 : Chatbot Streamlit persistant avec thèmes
# !pip install streamlit

import streamlit as st
import time

# Initialisation de l'historique
if 'messages' not in st.session_state:
    st.session_state.messages = []

# Sélecteur de thème (fonction de base, pas de switch natif Streamlit)
theme = st.radio("Choisissez le thème :", ("Clair", "Sombre"))
if theme == "Sombre":
    st.markdown(
        """
        <style>
        body {background-color: #111; color: #eee;}
        .stApp {background-color: #111; color: #eee;}
        </style>
        """,
        unsafe_allow_html=True,
    )

st.title("Chatbot persistant")

# Interface de chat
with st.form(key="chat_form"):
    user_input = st.text_input("Votre message", "")
    submit = st.form_submit_button("Envoyer")
    if submit and user_input:
        st.session_state.messages.append(("Utilisateur", user_input))
        with st.spinner("Laisse-moi réfléchir…"):
            time.sleep(1.5)
        bot_reply = f"Je réponds à: '{user_input}'"
        st.session_state.messages.append(("Bot", bot_reply))

# Affichage historique
for sender, message in st.session_state.messages:
    st.write(f"**{sender} :** {message}")

# Export .txt
if st.button("Exporter la conversation en .txt"):
    historique = "\n".join([f"{s}: {m}" for s, m in st.session_state.messages])
    st.download_button(
        label="Télécharger",
        data=historique,
        file_name="conversation.txt"
    )


# Exercice 3 : FastAPI + Gradio - Sélecteur d'outils avec backend en direct
Analyse

Nécessite de démarrer un backend FastAPI (endpoint /tool), logique d’analyse du message côté API.

Gradio en client HTTP qui interagit avec l’API.

Traitement conditionnel côté backend (détection de mot-clé, réponse adaptée).

Points critiques

Séparation stricte frontend/backend.

Gestion des erreurs (timeout, réponses inattendues).

Sécurité minimale (pas de gestion d’authentification demandée ici).

Approche de résolution

Côté FastAPI :

Créer endpoint POST /tool.

Récupérer message, détecter mot-clé (weather, news, translate), renvoyer outil choisi.

Côté Gradio :

Interface : textbox + bouton de soumission.

Appel HTTP au backend.

Affichage du résultat retourné.

In [ ]:
# Cellule 3a : FastAPI Backend (sauver ce code dans api.py puis lancer avec uvicorn api:app --reload)
# !pip install fastapi uvicorn

from fastapi import FastAPI, Request
from pydantic import BaseModel

app = FastAPI()

class Message(BaseModel):
    text: str

@app.post("/tool")
async def select_tool(msg: Message):
    text = msg.text.lower()
    if "weather" in text:
        tool = "weather"
    elif "news" in text:
        tool = "news"
    elif "translate" in text:
        tool = "translate"
    else:
        tool = "aucun outil trouvé"
    return {"outil": tool}


In [ ]:
# Cellule 3b : Interface Gradio (adapte l’URL si nécessaire)
# !pip install gradio requests

import gradio as gr
import requests

def call_api(message):
    url = "http://127.0.0.1:8000/tool"  # adapte selon ton FastAPI
    try:
        response = requests.post(url, json={"text": message})
        return response.json().get("outil", "Erreur de réponse")
    except Exception as e:
        return f"Erreur API : {e}"

with gr.Blocks() as demo:
    gr.Markdown("## Sélecteur d'outils via FastAPI")
    msg = gr.Textbox(label="Votre message")
    bouton = gr.Button("Soumettre")
    result = gr.Textbox(label="Outil sélectionné")
    bouton.click(call_api, msg, result)

demo.launch()


# Exercice 4 : Streamlit – Forme conditionnelle pour la collecte de commentaires
Analyse

Formulaire dynamique conditionnel, qui affiche un champ en fonction du choix oui/non.

Collecte d’info, validation, affichage du résultat après soumission.

Bonus : Persistance des réponses dans un fichier CSV avec Pandas.

Points critiques

Utilisation correcte de st.form (pas de soumission accidentelle).

Affichage conditionnel et structuré des champs.

Gestion des écritures concurrentes dans le fichier CSV.

Approche de résolution

Initialiser formulaire avec st.form().

Champ radio : « Avez-vous apprécié la leçon ? » (oui/non).

Si « oui » : champ commentaire libre.

Si « non » : champ « suggestions », curseur de note (par exemple 1-10).

À la soumission : afficher le feedback soumis, puis enregistrer ligne dans un fichier CSV.

Utiliser pandas pour la gestion du CSV (création du fichier si inexistant).

In [ ]:
# Cellule 4 : Formulaire conditionnel Streamlit
# !pip install streamlit pandas

import streamlit as st
import pandas as pd
from pathlib import Path

# Pour l’enregistrement CSV
CSV_PATH = Path("feedbacks.csv")

st.title("Formulaire de feedback sur la leçon")

with st.form("feedback_form"):
    avis = st.radio("Avez-vous apprécié la leçon ?", ["Oui", "Non"])
    commentaire = ""
    amelioration = ""
    note = None
    if avis == "Oui":
        commentaire = st.text_area("Votre commentaire")
    else:
        amelioration = st.text_area("Qu’est-ce qui pourrait être amélioré ?")
        note = st.slider("Votre note sur la leçon (1 = médiocre, 10 = excellent)", 1, 10, 5)
    submit = st.form_submit_button("Envoyer")

if submit:
    feedback = {
        "avis": avis,
        "commentaire": commentaire,
        "amelioration": amelioration,
        "note": note if note else ""
    }
    st.write("### Merci pour votre retour !")
    st.json(feedback)
    # Enregistrement local (bonus)
    df = pd.DataFrame([feedback])
    if not CSV_PATH.exists():
        df.to_csv(CSV_PATH, index=False, mode='w')
    else:
        df.to_csv(CSV_PATH, index=False, mode='a', header=False)


# 3. Conseils de robustesse et bonnes pratiques

Structuration du code : séparer les fonctions métier, éviter la duplication, garder une logique claire par fonctionnalité.

Initialisation de l’état : toujours tester que l’état session/gr.State/variables FastAPI sont bien initialisées au lancement.

Sécurité/Validation : même dans les démos, filtrer les entrées utilisateurs (inputs numériques, anti-injection pour le backend).

Conservation des logs/erreurs : prévoir un bloc try/except côté backend FastAPI et messages d’erreur utilisateur dans Gradio/Streamlit.

Compatibilité : tester sur les dernières versions stables de chaque framework (Streamlit, Gradio, FastAPI, pandas).

# 4. Résumé

Chaque exercice est typique de la stack “no backend classique” de l’IA appliquée :

Gradio pour la prototypage rapide multipage,

Streamlit pour des workflows plus riches et interactifs,

FastAPI pour exposer des endpoints robustes,

Pandas pour persister ou traiter des données côté back.

Une résolution solide passe par la compartimentation claire des responsabilités (UI vs backend), l’usage rigoureux des gestions d’état/session, et la validation systématique de chaque étape par des tests unitaires rapides.